# 00 Setup And Config
Einmalige Initialisierung: Stage, Pfadkontext, Systemcheck, `run_id`.

In [1]:
from pathlib import Path
import sys

RUN_STAGE = "smoke"  # smoke|mini|mid|full
PATHS_CONFIG = "configs/paths.local.yaml"  # alternativ: configs/paths.colab.yaml

def _find_root(start: Path) -> Path:
    for c in [start, *start.parents]:
        if (c / "src").exists() and (c / "configs").exists():
            return c.resolve()
    return start.resolve()

ROOT = _find_root(Path.cwd())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print("ROOT:", ROOT)

SEED_OVERRIDE = None
DEVICE = "auto"
USE_STUB_EMBEDDINGS = False
PREFER_PRECOMPUTED_ADS = True


ROOT: C:\Users\rapha\Documents\Studium\Promotionsstudium\MPIWG\2_Notebooks\Author_Name_Disambiguation


In [2]:
import os
import json
import uuid
import platform
import shutil
from datetime import datetime

import pandas as pd

try:
    from dotenv import load_dotenv
    load_dotenv(ROOT / ".env")
    print(".env loaded")
except Exception:
    print("python-dotenv nicht installiert - .env loading übersprungen")

print("HF_TOKEN gesetzt:", bool(os.getenv("HF_TOKEN", "")))


.env loaded
HF_TOKEN gesetzt: True


In [3]:
from src.common.config import load_notebook_context, build_run_dirs

CTX = load_notebook_context(run_stage=RUN_STAGE, project_root=ROOT, paths_config=PATHS_CONFIG)
DATA = CTX["DATA"]
ART = CTX["ART"]
MODEL = CTX["MODEL"]
RUN = CTX["RUN"]
if SEED_OVERRIDE is not None:
    RUN["seed"] = int(SEED_OVERRIDE)

cfg_preview = {
    "run_stage": RUN_STAGE,
    "run_seed": RUN.get("seed"),
    "subset_target_mentions": RUN.get("subset_target_mentions"),
    "raw_lspo": DATA["raw_lspo_parquet"],
    "raw_ads_publications": DATA["raw_ads_publications"],
    "raw_ads_references": DATA["raw_ads_references"],
    "model": MODEL.get("name"),
    "loss": MODEL.get("training", {}).get("loss"),
    "arch": f"{MODEL['training']['input_dim']}->{MODEL['training']['hidden_dim']}->{MODEL['training']['output_dim']}",
}

pd.DataFrame([cfg_preview]).T.rename(columns={0: "value"})


,value
run_stage,smoke
run_seed,11
subset_target_mentions,1000
raw_lspo,C:\Users\rapha\Documents\Studium\Promotionsstu...
raw_ads_publications,C:\Users\rapha\Documents\Studium\Promotionsstu...
raw_ads_references,C:\Users\rapha\Documents\Studium\Promotionsstu...
model,nand_best
loss,infonce
arch,818->1024->256


In [4]:
# Systemcheck
try:
    import psutil
    ram_gb = psutil.virtual_memory().total / (1024**3)
except Exception:
    ram_gb = None

free_gb = shutil.disk_usage(ROOT).free / (1024**3)

gpu_name = None
cuda_ok = False
try:
    import torch
    cuda_ok = torch.cuda.is_available()
    if cuda_ok:
        gpu_name = torch.cuda.get_device_name(0)
except Exception:
    pass

print("RAM_GB:", ram_gb)
print("FREE_DISK_GB:", round(free_gb, 2))
print("CUDA_AVAILABLE:", cuda_ok)
print("GPU:", gpu_name)
print("Python:", platform.python_version())


RAM_GB: 43.82659149169922
FREE_DISK_GB: 33.66
CUDA_AVAILABLE: False
GPU: None
Python: 3.10.13


In [5]:
RUN_ID = f"{RUN_STAGE}_{datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')}_{uuid.uuid4().hex[:8]}"
RUN_DIRS = build_run_dirs(DATA, ART, RUN_ID)
for p in RUN_DIRS.values():
    p.mkdir(parents=True, exist_ok=True)

context = {
    "run_id": RUN_ID,
    "run_stage": RUN_STAGE,
    "device": DEVICE,
    "use_stub_embeddings": USE_STUB_EMBEDDINGS,
    "prefer_precomputed_ads": PREFER_PRECOMPUTED_ADS,
}
with (RUN_DIRS["metrics"] / "00_context.json").open("w", encoding="utf-8") as f:
    json.dump(context, f, indent=2)

print("RUN_ID:", RUN_ID)
print(json.dumps({k: str(v) for k, v in RUN_DIRS.items()}, indent=2))


RUN_ID: smoke_20260212T173721Z_30edb99a
{
  "metrics": "C:\\Users\\rapha\\Documents\\Studium\\Promotionsstudium\\MPIWG\\2_Notebooks\\Author_Name_Disambiguation\\artifacts\\metrics\\smoke_20260212T173721Z_30edb99a",
  "checkpoints": "C:\\Users\\rapha\\Documents\\Studium\\Promotionsstudium\\MPIWG\\2_Notebooks\\Author_Name_Disambiguation\\artifacts\\checkpoints\\smoke_20260212T173721Z_30edb99a",
  "pair_scores": "C:\\Users\\rapha\\Documents\\Studium\\Promotionsstudium\\MPIWG\\2_Notebooks\\Author_Name_Disambiguation\\artifacts\\pair_scores\\smoke_20260212T173721Z_30edb99a",
  "clusters": "C:\\Users\\rapha\\Documents\\Studium\\Promotionsstudium\\MPIWG\\2_Notebooks\\Author_Name_Disambiguation\\artifacts\\clusters\\smoke_20260212T173721Z_30edb99a",
  "embeddings": "C:\\Users\\rapha\\Documents\\Studium\\Promotionsstudium\\MPIWG\\2_Notebooks\\Author_Name_Disambiguation\\artifacts\\embeddings\\smoke_20260212T173721Z_30edb99a",
  "subset_cache": "C:\\Users\\rapha\\Documents\\Studium\\Promotionsst